In [6]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
import sys
from pathlib import Path

project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root / "src"))

# Loss function experiments

Demonstrate `MSELoss_CP` on a synthetic regression dataset. First we build the data, then we check the loss against `torch.nn.MSELoss`, confirm the loss shrinks as predictions improve, and finally sweep a model parameter to show the loss forms a bowl with its minimum at the true value.

`MSELoss_CP.forward` computes `mean((prediction - target)**2)` and returns it as a scalar `Tensor_CP`. For 1-D targets this is the same value as `torch.nn.MSELoss` with its default `reduction="mean"`.

In [8]:
import numpy as np
import torch
from torch import nn

from tinytorch.loss_functions import MSELoss_CP
from tinytorch.tensor import Tensor_CP

np.random.seed(42)
torch.manual_seed(42)

print(f"NumPy:   {np.__version__}")
print(f"PyTorch: {torch.__version__}")
print("Device:  CPU")

NumPy:   2.5.0
PyTorch: 2.12.1+cpu
Device:  CPU


### Synthetic dataset

We generate a simple linear relationship `y = true_slope * x + true_bias` and add Gaussian noise so the targets are not perfectly linear. This mirrors a real regression problem where the model can never drive the loss to exactly zero.

In [9]:
rng = np.random.default_rng(7)

sample_count = 256
true_slope = 2.0
true_bias = -1.0
noise_std = 0.5

x = rng.uniform(-3.0, 3.0, size=sample_count).astype(np.float32)
noise = rng.normal(0.0, noise_std, size=sample_count).astype(np.float32)
y = (true_slope * x + true_bias + noise).astype(np.float32)

print(f"Samples: {sample_count}")
print(f"True relationship: y = {true_slope} * x + ({true_bias}) + noise(std={noise_std})")
print(f"x range: [{x.min():.2f}, {x.max():.2f}]")
print(f"y range: [{y.min():.2f}, {y.max():.2f}]")

Samples: 256
True relationship: y = 2.0 * x + (-1.0) + noise(std=0.5)
x range: [-2.98, 2.97]
y range: [-7.91, 5.06]


### Correctness against PyTorch

We use the noisy targets from the true parameters as the model prediction, so the only error left is the noise. `MSELoss_CP` should report the same loss as `torch.nn.MSELoss`.

In [10]:
def mse_cp(predictions, targets):
    """Mean squared error from MSELoss_CP (returns a scalar Tensor_CP)."""
    loss = MSELoss_CP().forward(Tensor_CP(predictions), Tensor_CP(targets))
    return float(loss.data)


predictions = (true_slope * x + true_bias).astype(np.float32)

loss_cp = mse_cp(predictions, y)
loss_torch = float(
    nn.MSELoss()(torch.from_numpy(predictions), torch.from_numpy(y))
)

print(f"TinyTorch MSELoss_CP: {loss_cp:.6f}")
print(f"PyTorch  nn.MSELoss: {loss_torch:.6f}")
print(f"Absolute difference:  {abs(loss_cp - loss_torch):.2e}")

assert np.isclose(loss_cp, loss_torch, atol=1e-5), "MSE loss mismatch!"
print("MSELoss_CP matches PyTorch")

TinyTorch MSELoss_CP: 0.231074
PyTorch  nn.MSELoss: 0.231074
Absolute difference:  1.49e-08
MSELoss_CP matches PyTorch


### The loss reflects prediction quality

A model whose predictions are close to the targets should produce a smaller loss than a poor one. We compare the true-parameter model against a deliberately wrong model and against the trivial baseline of always predicting the mean of `y`.

In [11]:
good_predictions = (true_slope * x + true_bias).astype(np.float32)
bad_predictions = (-1.0 * x + 4.0).astype(np.float32)
mean_baseline = np.full_like(y, y.mean())

loss_good = mse_cp(good_predictions, y)
loss_bad = mse_cp(bad_predictions, y)
loss_baseline = mse_cp(mean_baseline, y)

print(f"Good model (true params):       {loss_good:.4f}")
print(f"Mean baseline (predict y.mean): {loss_baseline:.4f}")
print(f"Bad model (wrong slope):        {loss_bad:.4f}")

assert loss_good < loss_baseline < loss_bad
print("Loss ranks the models as expected: good < baseline < bad")

Good model (true params):       0.2311
Mean baseline (predict y.mean): 12.1080
Bad model (wrong slope):        51.9029
Loss ranks the models as expected: good < baseline < bad


### Loss landscape over the slope

Holding the bias at its true value, we sweep candidate slopes and record the loss for each. The loss forms a convex bowl whose minimum sits at the slope that generated the data, which is exactly why minimizing MSE recovers the underlying parameters.

In [12]:
candidate_slopes = np.linspace(true_slope - 2.0, true_slope + 2.0, 9)

print(f"{'slope':>8} | {'MSE loss':>10}")
print("-" * 22)
losses = []
for slope in candidate_slopes:
    candidate_predictions = (slope * x + true_bias).astype(np.float32)
    loss = mse_cp(candidate_predictions, y)
    losses.append(loss)
    marker = "  <- true slope" if np.isclose(slope, true_slope) else ""
    print(f"{slope:>8.2f} | {loss:>10.4f}{marker}")

best_slope = candidate_slopes[int(np.argmin(losses))]
print(f"\nLowest loss at slope = {best_slope:.2f} (true slope = {true_slope})")

assert np.isclose(best_slope, true_slope), "Loss minimum should be at the true slope!"
print("MSE is minimized at the true slope")

   slope |   MSE loss
----------------------
    0.00 |    12.1081
    0.50 |     6.8894
    1.00 |     3.1703
    1.50 |     0.9509
    2.00 |     0.2311  <- true slope
    2.50 |     1.0109
    3.00 |     3.2904
    3.50 |     7.0695
    4.00 |    12.3483

Lowest loss at slope = 2.00 (true slope = 2.0)
MSE is minimized at the true slope


### Results

- `MSELoss_CP` returns the same mean squared error as `torch.nn.MSELoss` on the synthetic data.
- The loss correctly ranks a good model below the mean baseline and far below a wrong model.
- Sweeping the slope shows a convex loss bowl whose minimum coincides with the true data-generating slope, which is the property an optimizer relies on to learn.